In [5]:
import pandas as pd

# Load data
csf_nodes = pd.read_csv("./9. PUC/csf_nodes.csv") # columns: node, direction
ctx_nodes = pd.read_csv("./9. PUC/network_nodes.csv") # columns: ID, Mean Log2 Fold Change Direction (DSS)
edges = pd.read_csv("8. Filter FDR/final_edges_with_fdr_filtered.csv") # columns: Gene, Metabolite, Sign

# Build lookup maps
# Gene -> direction, from network_nodes
gene_direction_map = ctx_nodes.set_index("ID")["Mean Log2 Fold Change Direction (DSS)"].to_dict()

# Metabolite -> direction, from csf_nodes
metabolite_direction_map = csf_nodes.set_index("node")["direction"].to_dict()

# Map values onto edges
edges["ctx_direction"] = edges["Gene"].map(gene_direction_map)
edges["cpx_direction"] = edges["Metabolite"].map(metabolite_direction_map)

# Reorder so the new columns sit right next to their source columns
cols = list(edges.columns)
cols.remove("ctx_direction")
cols.remove("cpx_direction")

gene_idx = cols.index("Gene")
cols.insert(gene_idx + 1, "ctx_direction")

# recompute Metabolite's index since the list shifted after inserting ctx_direction
metabolite_idx = cols.index("Metabolite")
cols.insert(metabolite_idx + 1, "cpx_direction")

edges = edges[cols]

# Sanity check: report any genes/metabolites that didn't find a match
missing_genes = edges.loc[edges["ctx_direction"].isna(), "Gene"].unique()
missing_metabolites = edges.loc[edges["cpx_direction"].isna(), "Metabolite"].unique()
if len(missing_genes) > 0:
    print(f"Warning: {len(missing_genes)} gene(s) not found in network_nodes: {missing_genes}")
if len(missing_metabolites) > 0:
    print(f"Warning: {len(missing_metabolites)} metabolite(s) not found in csf_nodes: {missing_metabolites}")

# Step 5 ----------------------------------
edges["predicted_effect_sign"] = edges["ctx_direction"] * edges["cpx_direction"]

# Step 7 ----------------------------------
edges["within_model = pooled r"] = (
    edges["Pooled r"].fillna(edges["Sign"])
    .eq(edges["Sign"].map({"+": 1.0, "-": -1.0}))
) & edges["Consistent"]

edges["Sign"] = edges["Sign"].map({"+": 1.0, "-": -1.0})

# Step 8 ----------------------------------
edges["within-model = predicted"] = (
    edges["Sign"].eq(edges["predicted_effect_sign"])
) & edges["Consistent"].fillna(False)

# Step 9 ----------------------------------
edges["pooled = predicted"] = edges["Pooled r"].eq(edges["predicted_effect_sign"])

# Step 10 ----------------------------------
edges["all_signs_agree"] = (
    edges["within_model = pooled r"]
    == edges["within-model = predicted"]
) & (
    edges["within-model = predicted"]
    == edges["pooled = predicted"]
)

# Rename -----------------------------------
edges["Type_a"] = "CSF"
edges["Type_b"] = "CTX"

edges = edges.rename(columns={
    "Sign": "within_model_sign",
    "Metabolite": "CSF_name",
    "cpx_direction": "direction_CSF",
    "Type_a":"type_CSF",
    "Gene": "brain_name",
    "ctx_direction": "direction_brain",
    "Type_b":"type_brain",
})

# Select columns -----------------------------------
edges = edges[[
    "brain_name",
    "direction_brain",
    "type_brain",
    "CSF_name",
    "direction_CSF",
    "type_CSF",
    "VECPAC r",
    "LPS r",
    "DSS r",
    "Pooled r",
    "Consistent",
    "within_model_sign",
    "predicted_effect_sign",
    "within_model = pooled r",
    "within-model = predicted",
    "pooled = predicted",
    "all_signs_agree"
]]

# Save -----------------------------------
output_path = "9. PUC/final_edges_with_fdr_filtered_with_directions.csv"
edges.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(edges.head())

 '4,5-Dihydro-1H-pyrazole' 'Sulindac'
 '3-Hydroxymethyl-2-trimethylsilyloxypentane' 'L-Threonine' 'Luciferin'
 'N-Isobutyrylglycine' 'Octadecanoic acid' 'Chlorobenzilate '
 'l-Leucine, n-butoxycarbonyl-N-methyl-, hexadecyl ester'
 '2-Butene-1,4-diol' '2-Cyanobenzaldehyde' '5-Nitroso-8-quinolinol'
 'L-Ornithine' 'Benzeneacetic acid' 'Tyr-Val' 'Trimethylolpropane'
 'L-Glutamic acid' 'Adipoyl-L-carnitine'
 '1,2-Bis(trimethylsiloxy)cyclohexene' 'Aceturic acid'
 '2-Cyclohexen-1-one, 3-methyl-6-(1-methylethyl)' 'L-Serine' 'Lupinine'
 '6-Nitroveratryl alcohol' '2-(Ethylamino)ethanol, N-acetyl'
 '2,5-Dimethoxybenzonitrile' '2-Buten-1-ol' '2-Butenedioic acid'
 'Ethanolamine'
 '9Z-Octadecenoic acid, (2S)-3-[(hydroxymercaptophosphinyl)oxy]-2-methoxypropyl ester '
 'L-Pyroglutamic acid' 'Synephrine' 'Sarcosine' 'Diclofenac'
 'Tromethamine' '2-Propyl-1-pentanol' 'Formamide'
 '2,4,6-Cycloheptatrien-1-one' 'Norephedrine' 'Isobutanol' 'L-Norleucine'
 'Methyl lactate' 'Niacinamide'
 '3-(4,7-Dimethoxy-2